In [6]:
# multi_back.py
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from typing import TypedDict,List
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser
import os
import json
import io
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)


from langchain_groq import ChatGroq
model_text = ChatGroq(
    # model="meta-llama/llama-4-scout-17b-16e-instruct",
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.5,
)
model_parse=ChatGroq(
model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.5,
)
import re


c:\Users\DIL JAIN\Desktop\agentic\genai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:

def extract_information(image):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            image,
            """
            You are an ingredient detection system.

    Task:
    Identify ONLY the food ingredients that are clearly visible in the image.

    Rules:

    1. Do NOT guess ingredients that are not visible.
    2. Do NOT infer ingredients based on common recipes.
    3. Do NOT add spices, oil, salt, or seasonings unless they are clearly visible.
    4. If uncertain, do not include the ingredient.
    5. Include chopped, sliced, or partially visible ingredients if they can be identified 
    6. Return ONLY valid JSON.
    7. No markdown, no explanations, no extra text.

    Analyze the image now.

            """
        ]
    )
    clean_text = response.text.replace("```json", "").replace("```", "").strip()
    data = json.loads(clean_text)
    # ingredients = data["ingredients"]
    # return ingredients
    if isinstance(data, dict):
        return data.get("ingredients", [])
    elif isinstance(data, list):
        return data
    return []


In [59]:
text=["tomato","onion","paneer","apple","banana","ginger","milk","cashew","almonds","garam-masala","wheat-flour","rice"]

In [60]:
from langchain_core.prompts import PromptTemplate
from typing import TypedDict,List
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser

class recipe_schema(BaseModel):
    dish_name:str=Field(description="name of the recipe")
    full_recipe:List[str]=Field(description="detailed recipe step by step for the particular dish")
class multi_recipies(BaseModel):
    multi:List[recipe_schema]=Field(description="multiple dishes")
    
recipe_parser = PydanticOutputParser(pydantic_object=recipe_schema)
multi_dishes_parser=PydanticOutputParser(pydantic_object=multi_recipies)

def recipe_agent(data):
    ingredients=data
    prompt = PromptTemplate(
        template="""
    You are an expert chef.

    Your task is to create the best possible dish using the ingredients provided by the user.

    Ingredients:
    {ingredients}

    Instructions:
    note that you  have to provide the quantity used for all ingredients 
    1. Analyze the available ingredients carefully.
    2. Create 3-4 dish that can realistically be prepared using these ingredients.
    3. You may use common pantry items such as salt, water, oil, and basic spices if required, u can add it without any problem.
    4. Do not introduce major ingredients that are not present in the provided list.
    5. Generate a clear and detailed step-by-step recipe.
    
    6. Each step should be concise and actionable.
    7. Return your response ONLY in the specified JSON format.
    8. note that it is not necessary to use each ingredient in the particular recipe you may and you may not be but note that i want the bestest recipe

    {format_instructions}
    """,
        input_variables=["ingredients"],
        partial_variables={
            "format_instructions": multi_dishes_parser.get_format_instructions()
        }
    )
    chain=prompt|model_text|multi_dishes_parser
    result=chain.invoke({"ingredients":ingredients})
    return result
recipies=recipe_agent(text)

In [61]:
all=recipies.multi

In [62]:
all

[recipe_schema(dish_name='Paneer Tomato Masala', full_recipe=['Dice 200\u202fg paneer and set aside.', 'Finely chop 1 large onion (≈150\u202fg) and 3 medium tomatoes (≈300\u202fg).', 'Grate 1\u202ftsp fresh ginger.', 'Heat 2\u202ftbsp oil in a pan over medium heat; add the onion and sauté 3\u202fmin until translucent.', 'Add ginger and sauté 30\u202fsec.', 'Stir in chopped tomatoes, 1\u202ftsp salt, and 1\u202ftsp garam‑masala; cook 5‑6\u202fmin until tomatoes break down.', 'Pour in 1\u202fcup water, bring to a gentle boil, then reduce heat and simmer 5\u202fmin.', 'Add the paneer cubes, stir gently, and simmer 3\u202fmin to heat through.', 'Taste and adjust salt if needed; garnish with chopped cilantro if available and serve hot with rice or wheat‑flour roti.']),
 recipe_schema(dish_name='Cashew‑Almond Rice Pilaf', full_recipe=['Rinse 1\u202fcup (200\u202fg) rice until water runs clear; drain.', 'Slice 1 small onion (≈80\u202fg) and grate ½\u202ftsp ginger.', 'Heat 2\u202ftbsp oil in 

In [63]:
class ingredients(BaseModel):
    ingredients_name:str=Field(description="name of the ingredient")
    quantity:str=Field(description="the quantity of the ingredient")

class dish_ingredients(BaseModel):
    dish_name: str = Field(description="name of the dish")
    key_ingredients: List[ingredients] = Field(description="list of major ingredients with their quantity used in this dish ")

class all_dishes_ingredients(BaseModel):
    dishes: List[dish_ingredients] = Field(description="key ingredients for each dish")

ingredient_extractor_parser = PydanticOutputParser(pydantic_object=all_dishes_ingredients)

def ingredients_find_agent(all_recipes):
    all_items=[]
    for item in all_recipes:
      dish_name=(item.dish_name)
      full_recipe=" ".join(item.full_recipe)
      recipe=(dish_name+full_recipe)
      
      prompt = PromptTemplate(
        template="""

You are a professional chef and ingredient analyst.

You will be given a recipe.

Your task is to identify all major food ingredients used in the dish and estimate their quantities.

Recipe:
{recipe}

Rules:

1. Extract ONLY real food ingredients.
2. Do NOT include utensils, cookware, cooking actions, or cooking methods.
3. Do NOT include water.
4. Include oils, ghee, butter, salt, and spices only if they are clearly used in the recipe.
5. If an ingredient quantity is explicitly mentioned, use that quantity exactly.
6. If quantity is not mentioned, estimate a reasonable quantity based on a typical serving size.

IMPORTANT:

- Quantity MUST contain units.
- Use units such as:
  - g
  - kg
  - tbsp
  - tsp
  - ml

- Keep ingredient names clean and standardized.

Good examples:

Paneer -> "100g"
Onion -> "100g"
Tomato -> "200g"
Oil -> "1 tbsp (10g)"
Cumin Seeds -> "1 tsp (5g)"

Return ONLY valid JSON.

Example:

{{
  "dishes": [
    {{
      "dish_name": "Paneer Bhurji",
      "key_ingredients": [
        {{
          "ingredients_name": "Paneer",
          "quantity": "200g"
        }},
        {{
          "ingredients_name": "Onion",
          "quantity": "100g"
        }},
        {{
          "ingredients_name": "Tomato",
          "quantity": "150g"
        }},
        {{
          "ingredients_name": "Oil",
          "quantity": "1 tbsp (10g)"
        }}
      ]
    }}
  ]
}}

No markdown.
No explanation.
No extra text.

{format_instructions}
""",input_variables=["dishes_text"],partial_variables={"format_instructions": ingredient_extractor_parser.get_format_instructions()}
    )

      chain = prompt | model_text | ingredient_extractor_parser
      result = chain.invoke({"recipe":recipe})
      all_items.append(result)
    return all_items
    


In [64]:
all_ingre=ingredients_find_agent(all)


In [65]:
import pandas as pd
df=pd.read_csv("final.csv")
df.head(1)

,Ingredient_name,Category,calories,protein,fats,carbohydrates,fiber,vitamins,Minerals,Notes / Aliases
0,tomato,Vegetable,18,0.9,0.2,3.9,1.2,"Vitamin C, Vitamin A, Vitamin K","Potassium, Folate",NaN


In [66]:
def nutrition_calc(product):
    mask = df["Ingredient_name"].str.lower() == product.lower()

    if mask.any():
        return df.loc[mask].to_dict(orient="records")[0]

    return {
        "Ingredient_name": product,
        "Category": "Unknown",
        "calories": 0,
        "protein": 0,
        "fats": 0,
        "carbohydrates": 0,
        "fiber": 0,
        "vitamins": "",
        "Minerals": "",
        "Notes / Aliases": ""
    }

In [67]:
import re

def extract_quantity(quantity_text):
    quantity_text = quantity_text.lower().strip()

    match = re.search(r"\((\d+(?:\.\d+)?)\s*(g|ml)\)", quantity_text)
    if match:
        return float(match.group(1)), match.group(2)

    match = re.search(r"(\d+(?:\.\d+)?)\s*(g|ml)", quantity_text)
    if match:
        return float(match.group(1)), match.group(2)

    return 0, None

In [68]:
all_ingre


[all_dishes_ingredients(dishes=[dish_ingredients(dish_name='Paneer Tomato Masala', key_ingredients=[ingredients(ingredients_name='Paneer', quantity='200g'), ingredients(ingredients_name='Onion', quantity='150g'), ingredients(ingredients_name='Tomato', quantity='300g'), ingredients(ingredients_name='Ginger', quantity='1 tsp'), ingredients(ingredients_name='Oil', quantity='2 tbsp (20g)'), ingredients(ingredients_name='Salt', quantity='1 tsp (5g)'), ingredients(ingredients_name='Garam masala', quantity='1 tsp (5g)'), ingredients(ingredients_name='Cilantro', quantity='1 tbsp (5g)')])]),
 all_dishes_ingredients(dishes=[dish_ingredients(dish_name='Cashew‑Almond Rice Pilaf', key_ingredients=[ingredients(ingredients_name='Rice', quantity='200g'), ingredients(ingredients_name='Onion', quantity='80g'), ingredients(ingredients_name='Ginger', quantity='0.5 tsp (2.5g)'), ingredients(ingredients_name='Oil', quantity='2 tbsp (28g)'), ingredients(ingredients_name='Cashew halves', quantity='0.25 cup (3

In [72]:
import re
final_data=[]
for item in all_ingre:
    main_info={}
    main_info["dish_name"]=item.dishes[0].dish_name
    temp=(item.dishes[0].key_ingredients)
    all=[]
    for i in temp:
        ins={}
        ins["ingredient_name"]=(i.ingredients_name)
        ins["quantity"]=(i.quantity)
        quantityyy=i.quantity
        quantity, unit = extract_quantity(quantityyy)
        # function calling
        result=nutrition_calc(i.ingredients_name)
        ins["protein"] = (float(result["protein"]) * quantity) / 100
        ins["fats"] = (float(result["fats"]) * quantity) / 100
        ins["calories"] = (float(result["calories"]) * quantity) / 100
        ins["carbohydrates"] = (float(result["carbohydrates"]) * quantity) / 100
        ins["vitamins"]=result["vitamins"]
        
        
        all.append(ins)
    main_info["ingredients_info"]=all
    final_data.append(main_info)
        
resss=(final_data)
    
    

In [71]:
import re
final_data=[]
for item in all_ingre:
    main_info={}
    protein=0
    calories=0
    fats=0
    carbohydrates=0
    vitaminn=set()
    main_info["dish_name"]=item.dishes[0].dish_name
    temp=(item.dishes[0].key_ingredients)
    alll=[]
    for i in temp:
        ins={}
        ins["ingredient_name"]=(i.ingredients_name)
        ins["quantity"]=(i.quantity)
        quantityyy=i.quantity
        quantity, unit = extract_quantity(quantityyy)
        # function calling
        result=nutrition_calc(i.ingredients_name)
        calc_protein= (float(result["protein"]) * quantity) / 100
        calc_fats = (float(result["fats"]) * quantity) / 100
        calc_calories = (float(result["calories"]) * quantity) / 100
        calc_carbohydrates= (float(result["carbohydrates"]) * quantity) / 100
        calc_vitamins=result["vitamins"]
        ins["protein"] = calc_protein
        ins["fats"] = calc_fats
        ins["calories"] = calc_calories
        ins["carbohydrates"] = calc_carbohydrates
        ins["vitamins"]=calc_vitamins
        
        protein+=calc_protein
        calories+=calc_calories
        fats+=calc_fats
        carbohydrates+=calc_carbohydrates
        if isinstance(calc_vitamins, str):
            for v in calc_vitamins.split(","):
                v = v.strip()
                if v:
                    vitaminn.add(v)
        elif isinstance(calc_vitamins, list):
            for v in calc_vitamins:
                v = str(v).strip()
                if v:
                    vitaminn.add(v)
            
        alll.append(ins)
    vitamins = list(vitaminn)
    main_info["ingredients_info"]=alll
    main_info["overall_protein"]=protein
    main_info["overall_fats"]=fats
    main_info["overall_carbohydrates"]=carbohydrates
    main_info["overall_calories"]=calories
    main_info["overall_vitamins"]=vitamins
    final_data.append(main_info)
        
print(final_data)
    
    

[{'dish_name': 'Paneer Tomato Masala', 'ingredients_info': [{'ingredient_name': 'Paneer', 'quantity': '200g', 'protein': 36.6, 'fats': 41.6, 'calories': 530.0, 'carbohydrates': 2.4, 'vitamins': 'Vitamin A, Vitamin D, Vitamin B2'}, {'ingredient_name': 'Onion', 'quantity': '150g', 'protein': 1.65, 'fats': 0.15, 'calories': 60.0, 'carbohydrates': 13.95, 'vitamins': 'Vitamin C, Vitamin B6'}, {'ingredient_name': 'Tomato', 'quantity': '300g', 'protein': 2.7, 'fats': 0.6, 'calories': 54.0, 'carbohydrates': 11.7, 'vitamins': 'Vitamin C, Vitamin A, Vitamin K'}, {'ingredient_name': 'Ginger', 'quantity': '1 tsp', 'protein': 0.0, 'fats': 0.0, 'calories': 0.0, 'carbohydrates': 0.0, 'vitamins': 'Vitamin B6, Vitamin C'}, {'ingredient_name': 'Oil', 'quantity': '2 tbsp (20g)', 'protein': 0.0, 'fats': 0.0, 'calories': 0.0, 'carbohydrates': 0.0, 'vitamins': ''}, {'ingredient_name': 'Salt', 'quantity': '1 tsp (5g)', 'protein': 0.0, 'fats': 0.0, 'calories': 0.0, 'carbohydrates': 0.0, 'vitamins': nan}, {'in

In [1]:
final_data[0]

NameError: name 'final_data' is not defined

In [7]:
dish_info={
      "dish_name": "Paneer Bell Pepper Stir‑Fry",
      "full_recipe": [
        "Gather ingredients: 200 g paneer (cut into 1‑cm cubes), 1 large green bell pepper (seeded and sliced into strips), 1 medium red onion (thinly sliced), 1 tbsp vegetable oil, 1 tsp cumin seeds, 1/2 tsp turmeric powder, 1 tsp garam masala, 1 tsp lemon juice, 2 tbsp water, and salt to taste.",
        "Heat oil in a wok over medium heat; add cumin seeds and let them sizzle for 10 seconds.",
        "Add sliced red onion and sauté until translucent, about 2 minutes.",
        "Stir in bell‑pepper strips and cook for 3‑4 minutes, keeping them crisp‑tender.",
        "Add paneer cubes, turmeric, garam masala, and salt; toss gently to coat and cook for another 2 minutes.",
        "Pour in water and lemon juice, stir quickly, and cook for 1 minute until a light sauce forms.",
        "Turn off heat, transfer to a serving plate, and serve hot with roti or rice."
      ],
      "key_ingredients": [
        {
          "ingredient_name": "Paneer",
          "quantity": "200g"
        },
        {
          "ingredient_name": "green bell pepper",
          "quantity": "150g"
        },
        {
          "ingredient_name": "red onion",
          "quantity": "150g"
        },
        {
          "ingredient_name": "oil",
          "quantity": "1 tbsp (10g)"
        },
        {
          "ingredient_name": "cumin seeds",
          "quantity": "1 tsp (5g)"
        },
        {
          "ingredient_name": "turmeric powder",
          "quantity": "1/2 tsp (2.5g)"
        },
        {
          "ingredient_name": "garam masala",
          "quantity": "1 tsp (5g)"
        },
        {
          "ingredient_name": "lemon juice",
          "quantity": "1 tsp (5ml)"
        },
        {
          "ingredient_name": "salt",
          "quantity": "1/2 tsp (3g)"
        }
      ],
      "ingredients_info": [
        {
          "ingredient_name": "Paneer",
          "quantity": "200g",
          "protein": 36.6,
          "fats": 41.6,
          "calories": 530,
          "carbohydrates": 2.4,
          "vitamins": "Vitamin A, Vitamin D, Vitamin B2"
        },
        {
          "ingredient_name": "green bell pepper",
          "quantity": "150g",
          "protein": 1.35,
          "fats": 0.3,
          "calories": 30,
          "carbohydrates": 6.9,
          "vitamins": "Vitamin C, Vitamin B6, Vitamin A"
        },
        {
          "ingredient_name": "red onion",
          "quantity": "150g",
          "protein": 1.65,
          "fats": 0.15,
          "calories": 60,
          "carbohydrates": 13.95,
          "vitamins": "Vitamin C, Vitamin B6"
        },
        {
          "ingredient_name": "oil",
          "quantity": "1 tbsp (10g)",
          "protein": 0,
          "fats": 10,
          "calories": 88.4,
          "carbohydrates": 0,
          "vitamins": "Vitamin E, Vitamin K"
        },
        {
          "ingredient_name": "cumin seeds",
          "quantity": "1 tsp (5g)",
          "protein": 0.89,
          "fats": 1.115,
          "calories": 18.75,
          "carbohydrates": 2.21,
          "vitamins": "Vitamin B1, Vitamin B3, Vitamin E"
        },
        {
          "ingredient_name": "turmeric powder",
          "quantity": "1/2 tsp (2.5g)",
          "protein": 0.2425,
          "fats": 0.0825,
          "calories": 7.8,
          "carbohydrates": 1.6775,
          "vitamins": "Vitamin C, Vitamin E, Vitamin B6"
        },
        {
          "ingredient_name": "garam masala",
          "quantity": "1 tsp (5g)",
          "protein": 0.65,
          "fats": 0.75,
          "calories": 17.5,
          "carbohydrates": 2.5,
          "vitamins": "Vitamin A, Vitamin C, Vitamin K"
        },
        {
          "ingredient_name": "lemon juice",
          "quantity": "1 tsp (5ml)",
          "protein": 0.055,
          "fats": 0.015,
          "calories": 1.45,
          "carbohydrates": 0.465,
          "vitamins": "Vitamin C, Vitamin B6, Vitamin B5"
        },
        {
          "ingredient_name": "salt",
          "quantity": "1/2 tsp (3g)",
          "protein": 0,
          "fats": 0,
          "calories": 0,
          "carbohydrates": 0,
          "vitamins": ""
        }
      ],
      "overall_calories": 753.9,
      "overall_protein": 41.44,
      "overall_fats": 54.01,
      "overall_carbohydrates": 30.1,
      "overall_vitamins": [
        "Vitamin B6",
        "Vitamin B3",
        "Vitamin C",
        "Vitamin K",
        "Vitamin B1",
        "Vitamin B2",
        "Vitamin A",
        "Vitamin E",
        "Vitamin D",
        "Vitamin B5"
      ]
    }

In [ ]:
def recipe_chat(dish_info,user_query):
    prompt = PromptTemplate(
        template="""You are an expert chef assistant.

The user is asking about or wants to modify the following recipe:

{recipe_context}

User Request:
{user_query}

Instructions:
1. If the user wants to modify the recipe — provide the updated recipe steps clearly
2. If the user wants to substitute an ingredient — suggest the best substitute with quantity
3. If the user wants more details about a step — explain it clearly
4. If the user asks about quantity changes — adjust and confirm all affected steps
5. If the user asks a general question about the recipe — answer directly
6. Keep your response concise and practical
7. Do not make up ingredients that weren't in the original recipe unless user explicitly asks to add something new

Respond in plain text. Be direct and helpful.
""",
        input_variables=["recipe_context", "user_query"]
    )
    
    chain = prompt | model_text
    result = chain.invoke({
        "recipe_context": dish_info,
        "user_query": user_query
    })
    return result.content

In [9]:
evaluater_func(dish_info=dish_info,user_query="but here i wanted to use red bell pepper insted of gren bell")

'**Updated Recipe – Paneer Red‑Bell‑Pepper Stir‑Fry**\n\n---\n\n### Ingredient list (only the changed line is highlighted)\n\n- Paneer – 200\u202fg  \n- **Red bell pepper – 150\u202fg** (instead of green)  \n- Red onion – 150\u202fg  \n- Oil – 1\u202ftbsp (≈10\u202fg)  \n- Cumin seeds – 1\u202ftsp (≈5\u202fg)  \n- Turmeric powder – ½\u202ftsp (≈2.5\u202fg)  \n- Garam masala – 1\u202ftsp (≈5\u202fg)  \n- Lemon juice – 1\u202ftsp (≈5\u202fml)  \n- Salt – ½\u202ftsp (≈3\u202fg)  \n\n*(All other quantities remain the same.)*\n\n---\n\n### Revised cooking steps\n\n1. **Gather** the ingredients above.  \n2. Heat oil in a wok over medium heat; add cumin seeds and let them sizzle for about\u202f10\u202fseconds.  \n3. Add the sliced red onion and sauté until translucent, ~2\u202fminutes.  \n4. **Add the red‑bell‑pepper strips** (instead of green) and cook 3–4\u202fminutes, keeping them crisp‑tender.  \n5. Add paneer cubes, turmeric, garam masala, and salt; toss gently to coat and cook another 2